In [1]:
import torch 
import pandas as pd
import open3d as o3d
import numpy as np
from pathlib import Path 
from torch import Tensor
import MinkowskiEngine as ME
from tqdm.notebook import tqdm 


from opr.models.place_recognition import MinkLoc3D

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


/usr/local/lib/python3.10/dist-packages/MinkowskiEngine-0.5.4-py3.10-linux-x86_64.egg/MinkowskiEngine/__init__.py:36: UserWarning: The environment variable `OMP_NUM_THREADS` not set. MinkowskiEngine will automatically set `OMP_NUM_THREADS=16`. If you want to set `OMP_NUM_THREADS` manually, please export it on the command line before running a python script. e.g. `export OMP_NUM_THREADS=12; python your_program.py`. It is recommended to set it below 24.
  warnings.warn(
INFO:faiss.loader:Loading faiss with AVX2 support.
INFO:faiss.loader:Could not load library with AVX2 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx2'")
INFO:faiss.loader:Loading faiss.
INFO:faiss.loader:Successfully loaded faiss.
2025-09-04 14:03:43.222 | WARNING  | opr.models.place_recognition.pointmamba:<module>:16 - The 'pointmamba' package is not installed. Please install it manually if neccessary.


# DataReader

In [2]:
class DataReader:
    def __init__(
            self, csv_file: str | Path, lidar_scans_dir: str | Path
        ) -> None:
        """Initialize DataReader for pose-timestamped point cloud data.

        Args:
            csv_file (str | Path): Path to the CSV file containing pose and timestamp data.
            lidar_scans_dir (str | Path): Directory containing the lidar scans in PCD format.
            pointcloud_quantization_size (float): Size for quantizing the point cloud coordinates.
                Default is 0.1.
        Raises:
            FileNotFoundError: If the CSV file or lidar scans directory does not exist.
        """
        csv_file = Path(csv_file)
        if not csv_file.exists():
            raise FileNotFoundError(f"CSV file {csv_file} does not exist.")

        self.lidar_scans_dir = Path(lidar_scans_dir)
        if not self.lidar_scans_dir.exists():
            raise FileNotFoundError(f"Lidar scans directory {self.lidar_scans_dir} does not exist.")

        self.df = self.read_csv(csv_file)

        self.scans_list = []
        for scan_id in self.df['lidar_timestamp'].values:
            path = self.lidar_scans_dir / f"{scan_id:06d}.pcd"
            if not path.exists():
                raise FileNotFoundError(f"Missing scan file: {path}")
            self.scans_list.append(path)


    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int) -> dict[str, Tensor]:
        """Get the pose and point cloud data for a given index.

        Args:
            idx (int): Index of the data point to retrieve.
        Returns:
            dict: A dictionary containing:
                - pose (Tensor): The pose as a 7-element tensor [x, y, z, qx, qy, qz, qw].
                - pointcloud_lidar_coords (Tensor): The coordinates of the point cloud as an Nx3 tensor.
                - pointcloud_lidar_feats (Tensor): The features of the point cloud as an Nx1 tensor (intensity or ones).
        Raises:
            IndexError: If the index is out of range.
            ValueError: If the scan file is empty or has an unexpected format.
        """
        if idx < 0 or idx >= len(self.df):
            raise IndexError("Index out of range.")

        pose = self.df[["x", "y", "z", "qx", "qy", "qz", "qw"]].iloc[idx].to_numpy()
        scan_filepath = self.scans_list[idx]
        pc_coords, pc_feats = self.read_scan(scan_filepath)

        output_dict = {
            "pose": Tensor(pose),
            "pointcloud_lidar_coords": Tensor(pc_coords),
            "pointcloud_lidar_feats": Tensor(pc_feats)
        }

        return output_dict

    def read_scan(self, scan_filepath: str | Path) -> tuple[np.ndarray, np.ndarray]:
        """Read a point cloud scan from a file.
        Args:
            scan_filepath (str | Path): Path to the point cloud file.
        Returns:
            tuple: A tuple containing:
                - coordinates (np.ndarray): The coordinates of the point cloud as an Nx3 array.
                - features (np.ndarray): The features of the point cloud as an Nx1 array (intensity or ones).
        Raises:
            ValueError: If the scan file is empty or has an unexpected format.
        """
        scan = o3d.io.read_point_cloud(str(scan_filepath))
        if not scan.has_points():
            raise ValueError(f"Scan file {scan_filepath} is empty or invalid.")
        # Convert to numpy array for easier manipulation
        scan = np.asarray(scan.points)
        coordinates = scan[:, :3]  # Get the first three columns (x, y, z)
        if scan.shape[1] == 3:
            features = np.ones((coordinates.shape[0], 1))
        elif scan.shape[1] == 4:
            features = scan[:, 3:4]  # Get the fourth column (intensity)
        else:
            raise ValueError(f"Unexpected scan format with shape {scan.shape}. Expected 3 or 4 columns.")
        return coordinates, features

    def read_csv(self, filepath: str | Path) -> pd.DataFrame:
        """Read a CSV file containing pose and timestamp data.
        Args:
            filepath (str | Path): Path to the CSV file.
        Returns:
            pd.DataFrame: A DataFrame containing the pose and timestamp data.
        Raises:
            FileNotFoundError: If the CSV file does not exist.
        """
        dtype_mapping = {
            'pose_timestamp': np.int64,
            'lidar_timestamp': np.int64,
            'x': np.float64,
            'y': np.float64,
            'z': np.float64,
            'qx': np.float64,
            'qy': np.float64,
            'qz': np.float64,
            'qw': np.float64,
        }
        df = pd.read_csv(filepath, dtype=dtype_mapping)
        return df
    
def preprocess_input(input_data, pc_quantization_size):
        """Preprocess input data."""
        out_dict = {}
        for key in input_data:
            quantized_coords, quantized_feats = ME.utils.sparse_quantize(
                coordinates=input_data["pointcloud_lidar_coords"],
                features=input_data["pointcloud_lidar_feats"],
                quantization_size=pc_quantization_size,
            )
            out_dict["pointclouds_lidar_coords"] = ME.utils.batched_coordinates([quantized_coords])
            out_dict["pointclouds_lidar_feats"] = quantized_feats
        return out_dict



In [3]:
LOCAL_DATA_DIR = Path("/home/docker_mmpr/multimodal-place-recognition/data/2025-03-26-mmpr-datasets/")
DATASETS_ROOT = Path("/home/docker_mmpr/Datasets/")
SBER_OFFICE_DATA_DIR = DATASETS_ROOT / "2025-03-26-mmpr-datasets" / "keyframe-lidar-maps" / "keyframe-lidar-maps" /"mmpr_dataset" / "map1" / "keyframe_map"

query_reader = DataReader(
    csv_file=LOCAL_DATA_DIR / "query_lidar_frames.csv",
    lidar_scans_dir=SBER_OFFICE_DATA_DIR / "scans",
)

# Model

In [4]:
model = MinkLoc3D()
model = model.to('cuda')

# Compute resources

In [5]:
def pointcloud_extent(quantized_coords):
    """
    quantized_coords: torch.Tensor (N, 3) or (N, 4) if batched [batch_idx, x, y, z]
    Returns width, height, depth in voxel units.
    """
    if quantized_coords.shape[1] == 4:  # batched coords [b, x, y, z]
        coords = quantized_coords[:, 1:]  # drop batch index
    else:
        coords = quantized_coords

    mins = coords.min(dim=0).values
    maxs = coords.max(dim=0).values
    size = maxs - mins + 1  # +1 so single voxel has size 1
    return {
        "min": mins.tolist(),
        "max": maxs.tolist(),
        "width_x": int(size[0].item()),
        "height_y": int(size[1].item()),
        "depth_z": int(size[2].item()),
    }


def measure_gpu_memory(model, test_input, device="cuda"):
    torch.cuda.reset_peak_memory_stats(device)
    
    test_input = {k: v.to(device) for k, v in test_input.items()}
    model = model.to(device)
    
    with torch.no_grad():
        _ = model(test_input)
    
    torch.cuda.synchronize()
    allocated = torch.cuda.memory_allocated(device) / (1024 ** 2) 
    reserved = torch.cuda.memory_reserved(device) / (1024 ** 2)   
    peak = torch.cuda.max_memory_allocated(device) / (1024 ** 2)  
    
    return {"allocated_MiB": allocated, "reserved_MiB": reserved, "peak_MiB": peak}

def measure_averaged_gpu_memory(query_reader, model, pc_quantization_size):
    allocated = []
    reserved = []
    peak = []
    for q_idx, query in tqdm(enumerate(query_reader), total=len(query_reader)):
        test_input = preprocess_input(query_reader[0], pc_quantization_size)
        one_batch_result = measure_gpu_memory(model, test_input)
        allocated.append(one_batch_result['allocated_MiB'])
        reserved.append(one_batch_result['reserved_MiB'])
        peak.append(one_batch_result['peak_MiB'])

    print("==== Resources during model inference. Averaged over all queries ====")
    print(f"Quantization size: {pc_quantization_size}")
    for memory_type, values in zip(
        ["Allocated", "Reserved", "Peak"], 
        [allocated, reserved, peak]
    ):
        print(f"{memory_type} statistics:")
        print(f"\tMin: {np.min(values):.6} MiB")
        print(f"\tMax: {np.max(values):.6f} MiB")
        print(f"\tMedian: {np.median(values):.6f} MiB")
        print(f"\tMean: {np.mean(values):.3f} MiB")

In [6]:
pc_quantization_size = 0.01
measure_averaged_gpu_memory(query_reader, model, pc_quantization_size)

  0%|          | 0/1626 [00:00<?, ?it/s]

==== Resources during model inference. Averaged over all queries ====
Quantization size: 0.01
Allocated statistics:
	Min: 12.8711 MiB
	Max: 12.871094 MiB
	Median: 12.871094 MiB
	Mean: 12.871 MiB
Reserved statistics:
	Min: 192.0 MiB
	Max: 192.000000 MiB
	Median: 192.000000 MiB
	Mean: 192.000 MiB
Peak statistics:
	Min: 134.127 MiB
	Max: 134.232910 MiB
	Median: 134.232910 MiB
	Mean: 134.233 MiB


In [7]:
pc_quantization_size = 0.03
measure_averaged_gpu_memory(query_reader, model, pc_quantization_size)

  0%|          | 0/1626 [00:00<?, ?it/s]

==== Resources during model inference. Averaged over all queries ====
Quantization size: 0.03
Allocated statistics:
	Min: 12.7427 MiB
	Max: 12.742676 MiB
	Median: 12.742676 MiB
	Mean: 12.743 MiB
Reserved statistics:
	Min: 192.0 MiB
	Max: 192.000000 MiB
	Median: 192.000000 MiB
	Mean: 192.000 MiB
Peak statistics:
	Min: 66.626 MiB
	Max: 66.625977 MiB
	Median: 66.625977 MiB
	Mean: 66.626 MiB


In [8]:
pc_quantization_size = 0.05
measure_averaged_gpu_memory(query_reader, model, pc_quantization_size)

  0%|          | 0/1626 [00:00<?, ?it/s]

==== Resources during model inference. Averaged over all queries ====
Quantization size: 0.05
Allocated statistics:
	Min: 12.5952 MiB
	Max: 12.595215 MiB
	Median: 12.595215 MiB
	Mean: 12.595 MiB
Reserved statistics:
	Min: 192.0 MiB
	Max: 192.000000 MiB
	Median: 192.000000 MiB
	Mean: 192.000 MiB
Peak statistics:
	Min: 54.9922 MiB
	Max: 54.992188 MiB
	Median: 54.992188 MiB
	Mean: 54.992 MiB


In [9]:
pc_quantization_size = 0.1
measure_averaged_gpu_memory(query_reader, model, pc_quantization_size)

  0%|          | 0/1626 [00:00<?, ?it/s]

==== Resources during model inference. Averaged over all queries ====
Quantization size: 0.1
Allocated statistics:
	Min: 12.395 MiB
	Max: 12.395020 MiB
	Median: 12.395020 MiB
	Mean: 12.395 MiB
Reserved statistics:
	Min: 192.0 MiB
	Max: 192.000000 MiB
	Median: 192.000000 MiB
	Mean: 192.000 MiB
Peak statistics:
	Min: 35.4409 MiB
	Max: 35.440918 MiB
	Median: 35.440918 MiB
	Mean: 35.441 MiB


In [10]:
param_size = 0
for param in model.parameters():
    param_size += param.nelement() * param.element_size()
buffer_size = 0
for buffer in model.buffers():
    buffer_size += buffer.nelement() * buffer.element_size()

size_all_mb = (param_size + buffer_size) / 1024**2
print('model size: {:.3f} MiB'.format(size_all_mb))

model size: 4.031 MiB


In [11]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

total_buffers = sum(b.numel() for b in model.buffers())

print(f"Total parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")
print(f"Non-trainable params : {total_params - trainable_params:,}")
print(f"Buffers              : {total_buffers:,}")
print(f"Model size (params + buffers): {total_params + total_buffers:,}")

Total parameters     : 1,055,713
Trainable parameters : 1,055,713
Non-trainable params : 0
Buffers              : 1,099
Model size (params + buffers): 1,056,812
